# Fellegi-Sunter Baseline — Manual Validation Notebook

**Purpose.** Pull 5 random record pairs from each predicted tier (`auto_merge`, `human_review`, `no_match`) produced by `fs_splink_baseline` and manually inspect the match probability against the underlying records.

**Where this runs.** This notebook is authored off-VM but **executes only on the VM** against real `MDM_Population` data. Inputs are auto-resolved to the highest-versioned cleaned parquet + candidate-pairs parquet on disk (same convention as `run_real_baseline.py`).

**Output / artifact.** After **Run All**, the reviewer fills in the *Reviewer judgments* section at the bottom of this notebook (per-pair verdict + notes), saves, commits, pushes. **The committed notebook is the written validation record.**

**PHI note.** Output cells will contain identifier values (names, DOB, SSN, addresses). They stay on the VM. If you commit this notebook with outputs, you are committing PHI to the repo — decide deliberately whether to `Cell → All Output → Clear` before committing.

## 1. Setup & imports

In [ ]:
from __future__ import annotations

import re
import sys
from datetime import datetime
from pathlib import Path

# Project root = two levels up from notebooks/fellegi_sunter/.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from models.experiments.fs_splink_baseline import FSBaseline
from models.common.versioning import latest_versioned
from src.preprocessing.blocking import COL_PATID

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


In [ ]:
# Notebook-local constants. Override here if you want a different sample size,
# seed, or to point at alternate input directories.
RANDOM_SEED = 42
SAMPLES_PER_TIER = 5
U_MAX_PAIRS = 1e6  # Matches run_real_baseline.py production default.

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BLOCKING_DIR = PROJECT_ROOT / "data" / "blocking"

TIERS = ["auto_merge", "human_review", "no_match"]


## 2. Auto-resolve highest-versioned inputs

Mirrors the version-resolution convention of `src/preprocessing/clean.py` and `src/preprocessing/run_blocking.py`: parse the integer `v<N>` token from each filename and pick the maximum.

In [ ]:
cleaned_path = latest_versioned(PROCESSED_DIR, "MDM_Population_cleaned_v*_*.parquet")
pairs_path = latest_versioned(BLOCKING_DIR, "candidate_pairs_v*_*.parquet")

print(f"Cleaned parquet : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs : {pairs_path.relative_to(PROJECT_ROOT)}")


## 3. Score with `full_output=True`

`full_output=True` returns the classified frame *before* projection to the 5-col evaluation schema, retaining `match_probability`, `match_weight`, `classification_tier`, `source_blocks`, `n_blocks`, and Splink's `gamma_*` per-field agreement levels for downstream inspection.

Training is the slow step (single call to `run_fs_baseline`).

In [ ]:
df_clean = pd.read_parquet(cleaned_path)
print(f"Loaded {len(df_clean):,} cleaned records.")

In [ ]:
# FSBaseline.run() takes the candidate-pairs DataFrame directly (post E3 refactor).
df_pairs = pd.read_parquet(pairs_path)
df_scored = FSBaseline(u_max_pairs=U_MAX_PAIRS).run(
    candidate_pairs_df=df_pairs,
    df_clean=df_clean,
    full_output=True,
)

print(f"Scored {len(df_scored):,} candidate pairs.")
print("Tier breakdown:")
print(df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0))


## 4. Stratified random sample (5 per tier, fixed seed)

Deterministic across runs given `RANDOM_SEED = 42`. If a tier has fewer than `SAMPLES_PER_TIER` pairs, the notebook takes what's available and prints a warning rather than erroring.

In [ ]:
sampled_chunks = []
for tier in TIERS:
    tier_df = df_scored[df_scored["classification_tier"] == tier]
    n_available = len(tier_df)
    n_take = min(SAMPLES_PER_TIER, n_available)
    if n_take < SAMPLES_PER_TIER:
        print(
            f"WARN: tier {tier!r} has only {n_available} pairs; sampling {n_take}."
        )
    sampled_chunks.append(tier_df.sample(n=n_take, random_state=RANDOM_SEED))

sampled = pd.concat(sampled_chunks).reset_index(drop=True)
print(f"Sampled {len(sampled)} pairs total.")
print(sampled["classification_tier"].value_counts().reindex(TIERS, fill_value=0))

## 5. Side-by-side identifier display

For each sampled pair the notebook renders:

1. A one-line header with `classification_tier`, `match_probability`, `match_weight`, and which blocks fired.
2. A two-column DataFrame (`Record A` vs `Record B`) of the cleaned identifier fields. Values are pulled from the cleaned dataframe by `PATID` — this avoids depending on Splink's `_l/_r` suffix convention and shows all fields whether or not the model used them as evidence.
3. A compact view of Splink's `gamma_*` per-field agreement levels (when present), so you can see *why* the model assigned the score it did.

In [ ]:
# Fields to display per record, in order. (Label, cleaned-dataframe column name.)
DISPLAY_FIELDS: list[tuple[str, str]] = [
    ("PATID",              "PATID"),
    ("First name",         "FirstNM_clean"),
    ("Middle name",        "MiddleNM_clean"),
    ("Last name",          "LastNM_clean"),
    ("Full name tokens",   "full_name_tokens"),
    ("DOB",                "BirthDT_clean"),
    ("SSN (full)",         "SSN_clean"),
    ("SSN last-4",         "last_4_SSN"),
    ("Email",              "Email_clean"),
    ("Address line 1",     "AddressLine1_clean"),
    ("Address line 2",     "AddressLine2_clean"),
    ("City",               "CityNM_clean"),
    ("State",              "StateCD_clean"),
    ("ZIP",                "ZipCD_clean_base"),
    ("Phones (set)",       "Phones_set"),
]

# Index df_clean by PATID once for O(1) lookups.
_clean_indexed = df_clean.set_index(COL_PATID, drop=False)


def _lookup(patid: str) -> pd.Series:
    """Pull one cleaned record by PATID; returns an empty Series if absent."""
    try:
        return _clean_indexed.loc[patid]
    except KeyError:
        return pd.Series(dtype=object)


def render_identifier_table(patid_a: str, patid_b: str) -> pd.DataFrame:
    """Two-column side-by-side DataFrame of cleaned identifier fields."""
    rec_a = _lookup(patid_a)
    rec_b = _lookup(patid_b)
    rows = {}
    for label, col in DISPLAY_FIELDS:
        if col not in df_clean.columns:
            continue  # field absent in this cleaned parquet version; skip.
        rows[label] = [rec_a.get(col, np.nan), rec_b.get(col, np.nan)]
    return pd.DataFrame.from_dict(
        rows, orient="index", columns=["Record A", "Record B"]
    )


def render_gamma_table(row: pd.Series) -> pd.DataFrame | None:
    """Splink's gamma_<field> per-field agreement levels, if retained."""
    gamma_cols = [c for c in row.index if c.startswith("gamma_")]
    if not gamma_cols:
        return None
    out = pd.DataFrame(
        {"agreement_level": [row[c] for c in gamma_cols]},
        index=[c.removeprefix("gamma_") for c in gamma_cols],
    )
    out.index.name = "comparison"
    return out

In [ ]:
from IPython.display import display, Markdown

for i, row in sampled.iterrows():
    pair_num = i + 1
    patid_a = row["PATID_A"]
    patid_b = row["PATID_B"]
    tier = row["classification_tier"]
    p = row.get("match_probability", float("nan"))
    w = row.get("match_weight", float("nan"))
    src_blocks = row.get("source_blocks", "")
    n_blocks = row.get("n_blocks", "")

    display(Markdown(
        f"### Pair {pair_num}/{len(sampled)} \u2014 tier=`{tier}`  \n"
        f"`PATID_A={patid_a}` \u2194 `PATID_B={patid_b}`  \n"
        f"`match_probability={p:.4f}`  \u00b7  `match_weight={w:.3f}`  "
        f"\u00b7  `n_blocks={n_blocks}`  \u00b7  `source_blocks={src_blocks}`"
    ))
    display(render_identifier_table(patid_a, patid_b))
    gammas = render_gamma_table(row)
    if gammas is not None:
        display(Markdown("**Splink agreement levels (`gamma_*`):**"))
        display(gammas)

## 6. Reviewer judgment templates

Run the cell below to generate one markdown template per sampled pair. Copy the printed block into the **Reviewer judgments** markdown cell at the bottom of this notebook and fill in `Reviewer verdict` (`true_match` / `not_match` / `unsure`) and `Reviewer notes` for each pair. Save, commit, push.

Why a printed block (not interactive widgets): markdown survives `nbconvert`, diffs cleanly in git, and the committed notebook is then a self-contained validation record.

In [ ]:
template_lines = []
for i, row in sampled.iterrows():
    pair_num = i + 1
    template_lines.append(
        f"### Pair {pair_num}/{len(sampled)} \u2014 "
        f"PATID_A={row['PATID_A']}, PATID_B={row['PATID_B']}\n"
        f"\n"
        f"- **Predicted tier:** `{row['classification_tier']}`\n"
        f"- **Model match_probability:** {row.get('match_probability', float('nan')):.4f}\n"
        f"- **Reviewer verdict:** [ true_match | not_match | unsure ]\n"
        f"- **Reviewer notes:**\n"
        f"  - \n"
    )

print("\n".join(template_lines))

## 7. Diagnostic summary (non-PHI)

Provenance trail recording which dataset version was validated. Safe to commit even when output cells are cleared.

In [ ]:
tier_counts = df_scored["classification_tier"].value_counts().reindex(TIERS, fill_value=0)
total = int(tier_counts.sum())

print(f"Executed at         : {datetime.now().isoformat(timespec='seconds')}")
print(f"Cleaned parquet     : {cleaned_path.relative_to(PROJECT_ROOT)}")
print(f"Candidate pairs     : {pairs_path.relative_to(PROJECT_ROOT)}")
print(f"Cleaned records     : {len(df_clean):,}")
print(f"Candidate pairs     : {len(df_scored):,}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Samples per tier    : {SAMPLES_PER_TIER}")
print()
print("Tier distribution (full population):")
for tier in TIERS:
    n = int(tier_counts[tier])
    pct = (n / total * 100) if total else 0.0
    print(f"  {tier:<14s} {n:>10,}  ({pct:5.2f}%)")

## 8. Visual summaries

Population-level views of the baseline's output. All three figures share one palette and styling so they can drop directly into a presentation deck. PNGs are saved to `notebooks/fellegi_sunter/figures/` (gitignored) and filename-versioned by the resolved cleaned-parquet tag so figures don't overwrite across data refreshes.

**Why log scale on the histogram.** Real eMPI score distributions are bimodal: most candidate pairs collapse to ~0 (clear non-matches) or ~1 (clear matches), with a small `human_review` middle band. On a linear y-axis that middle band is visually invisible at production pair counts. Log scale keeps every tier legible without distorting the bimodal shape.

In [ ]:
import re as _re
import matplotlib.pyplot as plt

# ---- Presentation-grade styling (applies to all three figures) -------------
plt.rcParams.update({
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.labelcolor":    "#333333",
    "axes.edgecolor":     "#555555",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         "#DDDDDD",
    "grid.linewidth":     0.6,
    "grid.alpha":         0.8,
    "xtick.color":        "#333333",
    "ytick.color":        "#333333",
    "legend.frameon":     False,
    "savefig.dpi":        200,
    "savefig.bbox":       "tight",
    "savefig.facecolor":  "white",
})

# Shared palette — aligned with the reference strip chart.
TIER_COLORS = {
    "no_match":     "#B5B5B5",  # neutral gray
    "human_review": "#F0BE7E",  # warm peach
    "auto_merge":   "#88B888",  # muted green
}
TIER_EDGE = {
    "no_match":     "#7F7F7F",
    "human_review": "#C68A3F",
    "auto_merge":   "#4F8A4F",
}
# Display order: low → high match_probability (matches strip chart left→right).
TIER_ORDER_LOW_TO_HIGH = ["no_match", "human_review", "auto_merge"]

# Pull thresholds from the FS module so charts auto-track any retuning.
# Pull thresholds from FSBaseline so charts auto-track any retuning.
_baseline_cfg = FSBaseline().classification_config
REVIEW_FLOOR = _baseline_cfg.review_floor
AUTO_MERGE_THRESHOLD = _baseline_cfg.auto_merge_threshold

# Output dir for PNGs (gitignored via notebooks/**/figures/).
FIGURES_DIR = Path.cwd() / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Version tag pulled from the resolved cleaned parquet so PNG filenames track
# the dataset version they were generated from.
_version_match = _re.search(r"(v\d+_\d{4}_\d{2}_\d{2})", cleaned_path.name)
VERSION_TAG = _version_match.group(1) if _version_match else "unversioned"

# Pre-compute reusable stats.
_scores = df_scored["match_probability"].to_numpy()
_total = int(len(df_scored))
_tier_counts = df_scored["classification_tier"].value_counts().reindex(
    TIER_ORDER_LOW_TO_HIGH, fill_value=0
).astype(int)
_tier_pct = (_tier_counts / _total * 100) if _total else _tier_counts * 0.0
_five_num = {
    "min":    float(np.min(_scores)),
    "p25":    float(np.quantile(_scores, 0.25)),
    "median": float(np.median(_scores)),
    "p75":    float(np.quantile(_scores, 0.75)),
    "max":    float(np.max(_scores)),
}

print(f"Figures will save to : {FIGURES_DIR.relative_to(PROJECT_ROOT)}")
print(f"Version tag          : {VERSION_TAG}")


### 8.1 Predicted tier breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.2))

y_pos = np.arange(len(TIER_ORDER_LOW_TO_HIGH))
counts = _tier_counts.values
colors = [TIER_COLORS[t] for t in TIER_ORDER_LOW_TO_HIGH]
edges = [TIER_EDGE[t] for t in TIER_ORDER_LOW_TO_HIGH]

bars = ax.barh(y_pos, counts, color=colors, edgecolor=edges, linewidth=1.0, height=0.62)

# Annotate each bar with count + pct, placed just outside the bar end.
_max_count = int(max(counts)) if len(counts) else 0
x_pad = _max_count * 0.012 if _max_count > 0 else 0.5
for bar, tier in zip(bars, TIER_ORDER_LOW_TO_HIGH):
    n = int(_tier_counts[tier])
    pct = float(_tier_pct[tier])
    ax.text(
        bar.get_width() + x_pad,
        bar.get_y() + bar.get_height() / 2,
        f"{n:,}  ({pct:.2f}%)",
        va="center", ha="left", fontsize=10.5, color="#222222",
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(TIER_ORDER_LOW_TO_HIGH, fontsize=11)
ax.invert_yaxis()  # auto_merge on top (top-down reading: highest probability tier first)
ax.set_xlabel("Candidate pairs (count)")
ax.set_xlim(0, _max_count * 1.18 if _max_count > 0 else 1)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)

ax.set_title("Predicted Tier Breakdown", loc="left", pad=18)
ax.text(
    0.0, 1.04,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout()
out_path = FIGURES_DIR / f"tier_breakdown__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Tier counts:")
for tier in TIER_ORDER_LOW_TO_HIGH:
    print(f"  {tier:<14s} {int(_tier_counts[tier]):>10,}  ({float(_tier_pct[tier]):5.2f}%)")


### 8.2 Score distribution — five-number summary vs. classification thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))

# ---- Threshold bands (background) -----------------------------------------
ax.axvspan(0.0, REVIEW_FLOOR, color=TIER_COLORS["no_match"], alpha=0.55, lw=0)
ax.axvspan(REVIEW_FLOOR, AUTO_MERGE_THRESHOLD, color=TIER_COLORS["human_review"], alpha=0.55, lw=0)
ax.axvspan(AUTO_MERGE_THRESHOLD, 1.0, color=TIER_COLORS["auto_merge"], alpha=0.55, lw=0)

# ---- Threshold lines + top labels -----------------------------------------
for x, label in [(REVIEW_FLOOR, "review_floor"), (AUTO_MERGE_THRESHOLD, "auto_merge")]:
    ax.axvline(x, color="#555555", linestyle="--", linewidth=1.1)
    ax.text(x, 1.02, label, transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=10, color="#444444")
    ax.text(x, 0.96, f"{x:.2f}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9.5, color="#444444")

# ---- Tier % labels centered inside each band ------------------------------
band_centers = [
    ("no_match",     REVIEW_FLOOR / 2),
    ("human_review", (REVIEW_FLOOR + AUTO_MERGE_THRESHOLD) / 2),
    ("auto_merge",   (AUTO_MERGE_THRESHOLD + 1.0) / 2),
]
for tier, x_center in band_centers:
    ax.text(x_center, 0.28, tier, ha="center", va="center",
            fontsize=11, fontweight="semibold", color="#333333")
    ax.text(x_center, 0.14, f"({float(_tier_pct[tier]):.1f}%)",
            ha="center", va="center", fontsize=10, color="#555555")

# ---- Five-number summary dots + labels ------------------------------------
# Upper track keeps dots clear of band labels below.
dot_y = 0.70

# Handle the common case where min==p25 (both 0) or max==p75 (both 1) by
# stacking the two collided labels vertically so they don't overprint.
_left_collide = abs(_five_num["min"] - _five_num["p25"]) < 1e-6
_right_collide = abs(_five_num["max"] - _five_num["p75"]) < 1e-6

ax.scatter([_five_num["min"]], [dot_y], s=46, color="#111111", zorder=5)
if _left_collide:
    ax.text(_five_num["min"] + 0.012, dot_y + 0.08,
            f"min={_five_num['min']:g}", ha="left", va="bottom",
            fontsize=9.5, color="#222222")
    ax.text(_five_num["p25"] + 0.012, dot_y - 0.08,
            f"p25={_five_num['p25']:g}", ha="left", va="top",
            fontsize=9.5, color="#222222")
else:
    ax.text(_five_num["min"] + 0.012, dot_y,
            f"min={_five_num['min']:g}", ha="left", va="center",
            fontsize=9.5, color="#222222")
    ax.scatter([_five_num["p25"]], [dot_y], s=46, color="#111111", zorder=5)
    ax.text(_five_num["p25"] + 0.012, dot_y,
            f"p25={_five_num['p25']:g}", ha="left", va="center",
            fontsize=9.5, color="#222222")

# Median (always its own dot; label above to avoid the dot itself).
ax.scatter([_five_num["median"]], [dot_y], s=46, color="#111111", zorder=5)
ax.text(_five_num["median"], dot_y + 0.10,
        f"median\n{_five_num['median']:.4f}",
        ha="center", va="bottom", fontsize=9.5, color="#222222")

ax.scatter([_five_num["max"]], [dot_y], s=46, color="#111111", zorder=5)
if _right_collide:
    ax.text(_five_num["max"] - 0.012, dot_y + 0.08,
            f"max={_five_num['max']:g}", ha="right", va="bottom",
            fontsize=9.5, color="#222222")
    ax.text(_five_num["p75"] - 0.012, dot_y - 0.08,
            f"p75={_five_num['p75']:g}", ha="right", va="top",
            fontsize=9.5, color="#222222")
else:
    ax.text(_five_num["max"] - 0.012, dot_y,
            f"max={_five_num['max']:g}", ha="right", va="center",
            fontsize=9.5, color="#222222")
    ax.scatter([_five_num["p75"]], [dot_y], s=46, color="#111111", zorder=5)
    ax.text(_five_num["p75"] - 0.012, dot_y,
            f"p75={_five_num['p75']:g}", ha="right", va="center",
            fontsize=9.5, color="#222222")

# ---- Axes cosmetics -------------------------------------------------------
ax.set_xlim(-0.005, 1.005)
ax.set_ylim(0, 1)
ax.set_xlabel("match_probability (score)")
ax.set_yticks([])
ax.grid(False)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)

ax.set_title(
    "Score Distribution — Five-Number Summary vs. Classification Thresholds",
    loc="left", pad=22,
)
ax.text(
    0.0, 1.13,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout()
out_path = FIGURES_DIR / f"score_distribution_strip__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Five-number summary of match_probability:")
for k, v in _five_num.items():
    print(f"  {k:<6s} {v:.6f}")


### 8.3 Score histogram — match_probability colored by tier

In [ ]:
HIST_BINS = 50

fig, ax = plt.subplots(figsize=(11, 4.6))

bin_edges = np.linspace(0.0, 1.0, HIST_BINS + 1)
bin_width = bin_edges[1] - bin_edges[0]
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Build per-tier histograms over identical bins, then stack.
tier_hists = {}
for tier in TIER_ORDER_LOW_TO_HIGH:
    scores_t = df_scored.loc[df_scored["classification_tier"] == tier, "match_probability"].to_numpy()
    h, _ = np.histogram(scores_t, bins=bin_edges)
    tier_hists[tier] = h

cumulative = np.zeros(HIST_BINS, dtype=float)
for tier in TIER_ORDER_LOW_TO_HIGH:
    h = tier_hists[tier]
    ax.bar(
        bin_centers, h, width=bin_width * 0.95,
        bottom=cumulative,
        color=TIER_COLORS[tier], edgecolor=TIER_EDGE[tier], linewidth=0.5,
        label=f"{tier}  ({int(_tier_counts[tier]):,})",
        align="center",
    )
    cumulative = cumulative + h

# Log scale with a small floor so empty bins don't visually clip.
ax.set_yscale("log")
_max_total = max(cumulative.max(), 1)
ax.set_ylim(0.5, _max_total * 3)

# Threshold lines + top labels.
for x, label in [(REVIEW_FLOOR, "review_floor"), (AUTO_MERGE_THRESHOLD, "auto_merge")]:
    ax.axvline(x, color="#444444", linestyle="--", linewidth=1.1, zorder=3)
    ax.text(x, 1.02, f"{label} = {x:.2f}", transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=10, color="#444444")

ax.set_xlim(-0.01, 1.01)
ax.set_xlabel("match_probability (score)")
ax.set_ylabel("Candidate pairs (count, log scale)")
ax.grid(axis="x", visible=False)
ax.grid(axis="y", visible=False)

ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3,
    fontsize=10, handlelength=1.4, handleheight=1.0, borderpad=0.6,
)

ax.set_title(
    "Score Histogram — match_probability by Tier",
    loc="left", y=1.18, pad=0,
)
ax.text(
    0.0, 1.10,
    f"Fellegi-Sunter Splink baseline  ·  {VERSION_TAG}  ·  n = {_total:,} pairs  ·  {HIST_BINS} bins",
    transform=ax.transAxes, fontsize=10, color="#666666",
)

fig.tight_layout(rect=[0, 0, 1, 0.88])
out_path = FIGURES_DIR / f"score_histogram__{VERSION_TAG}.png"
fig.savefig(out_path)
plt.show()
print(f"Saved → {out_path.relative_to(PROJECT_ROOT)}")
print()
print("Five-number summary of match_probability:")
for k, v in _five_num.items():
    print(f"  {k:<6s} {v:.6f}")
print()
print("Per-tier counts (legend mirrors these):")
for tier in TIER_ORDER_LOW_TO_HIGH:
    print(f"  {tier:<14s} {int(_tier_counts[tier]):>10,}  ({float(_tier_pct[tier]):5.2f}%)")


## 9. Threshold-band sampling deep-dives

Two targeted sampling passes that feed the threshold-tuning conversation (separate from the Section-5 stratified sample, which served the broader sanity-check goal):

- **§9.1** — A 20-pair random sample drawn from the `human_review` tier only, to characterize what kinds of identifier discrepancies land in that middle band.
- **§9.2** — A 12-pair-per-band stratified sample drawn from the score bands surrounding each tier boundary (`[0.45, 0.55)` around `review_floor`, `[0.85, 0.95)` around `auto_merge`), to probe whether the current thresholds are well placed.

Both sections render the same identifier-table + `gamma_*` view as Section 5 and emit reviewer-judgment templates at the end so a manual reviewer can record verdicts in-line. Pairs already shown in §9.1 are deduplicated out of §9.2.

### 9.1 Human-review band characterization (20-pair random sample)

Goal: characterize what kinds of pair-discrepancies actually land in `human_review` (the small middle band — 1,369 pairs on the v3/v4 baseline, ~0.67% of all scored pairs). A 20-pair sample is enough to spot dominant failure modes without committing to a full census.

Reuses `render_identifier_table()` and `render_gamma_table()` from Section 5 so the rendered output is identical in shape to the 15-pair stratified sample reviewed in Section 5 — just sourced from `human_review` only.

In [ ]:
# §9.1 human_review band — random 20-pair sample for hand review.
PHASE4_HR_SAMPLE_N = 20

_hr_pool = df_scored[df_scored["classification_tier"] == "human_review"]
_hr_take = min(PHASE4_HR_SAMPLE_N, len(_hr_pool))
if _hr_take < PHASE4_HR_SAMPLE_N:
    print(f"WARN: human_review has only {len(_hr_pool)} pairs; sampling {_hr_take}.")
_hr_sample = _hr_pool.sample(n=_hr_take, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"§9.1 human_review sample: {len(_hr_sample)} pairs (seed={RANDOM_SEED})")
print()

for i, row in _hr_sample.iterrows():
    patid_a, patid_b = row["PATID_A"], row["PATID_B"]
    p = row.get("match_probability", float("nan"))
    w = row.get("match_weight", float("nan"))
    src_blocks = row.get("source_blocks", "")
    n_blocks = row.get("n_blocks", "")
    display(Markdown(
        f"#### Pair {i+1}/{len(_hr_sample)} — tier=`{row['classification_tier']}`  \n"
        f"`PATID_A={patid_a}` ↔ `PATID_B={patid_b}`  \n"
        f"`match_probability={p:.4f}`  ·  `match_weight={w:.3f}`  ·  "
        f"`n_blocks={n_blocks}`  ·  `source_blocks={src_blocks}`"
    ))
    display(render_identifier_table(patid_a, patid_b))
    gammas = render_gamma_table(row)
    if gammas is not None:
        display(gammas)


### 9.2 Threshold-boundary stratified samples (12 per band)

Two bands:

- `review_floor_band` — `score ∈ [0.45, 0.55)`: pairs straddling the `no_match` ↔ `human_review` threshold.
- `auto_merge_band` — `score ∈ [0.85, 0.95)`: pairs straddling the `human_review` ↔ `auto_merge` threshold.

Goal: judge by hand whether borderline pairs look more like the tier above or below the current threshold. Drives the decision on whether to retune `DEFAULT_AUTO_MERGE_THRESHOLD` (0.90) or `DEFAULT_REVIEW_FLOOR` (0.50).

Pairs already shown in §9.1 are deduplicated out of this sample so a reviewer never sees the same pair twice.

In [ ]:
# §9.2 stratified threshold-boundary sample (10-15 per band, dedup against §9.1).
PHASE4_PER_BAND = 12  # mid-point of the requested 10-15

_phase4_seen = set(zip(_hr_sample["PATID_A"], _hr_sample["PATID_B"])) if "_hr_sample" in dir() else set()
_scored_avail = df_scored[~df_scored.set_index(["PATID_A", "PATID_B"]).index.isin(_phase4_seen)]

def _band_sample(low: float, high: float) -> pd.DataFrame:
    mask = (_scored_avail["match_probability"] >= low) & (_scored_avail["match_probability"] < high)
    band = _scored_avail.loc[mask]
    n_take = min(PHASE4_PER_BAND, len(band))
    if n_take < PHASE4_PER_BAND:
        print(f"WARN: band [{low}, {high}) has only {len(band)} pairs; sampling {n_take}.")
    return band.sample(n=n_take, random_state=RANDOM_SEED).reset_index(drop=True)

_floor_sample = _band_sample(0.45, 0.55)
_auto_sample = _band_sample(0.85, 0.95)

print(f"§9.2 review_floor_band [0.45, 0.55): {len(_floor_sample)} pairs")
print(f"§9.2 auto_merge_band  [0.85, 0.95): {len(_auto_sample)} pairs")
print()

for _label, _frame in [
    ("review_floor_band [0.45, 0.55)", _floor_sample),
    ("auto_merge_band [0.85, 0.95)", _auto_sample),
]:
    display(Markdown(f"### §9.2 — {_label}"))
    for i, row in _frame.iterrows():
        patid_a, patid_b = row["PATID_A"], row["PATID_B"]
        p = row.get("match_probability", float("nan"))
        w = row.get("match_weight", float("nan"))
        src_blocks = row.get("source_blocks", "")
        display(Markdown(
            f"#### Pair {i+1}/{len(_frame)} — tier=`{row['classification_tier']}`  \n"
            f"`PATID_A={patid_a}` ↔ `PATID_B={patid_b}`  \n"
            f"`match_probability={p:.4f}`  ·  `match_weight={w:.3f}`  ·  "
            f"`source_blocks={src_blocks}`"
        ))
        display(render_identifier_table(patid_a, patid_b))
        gammas = render_gamma_table(row)
        if gammas is not None:
            display(gammas)


In [ ]:
# Reviewer-judgment templates for the §9 samples. Copy the printed block into
# the Reviewer judgments markdown cell at the bottom of this notebook, beneath
# the Section-6 templates.

_phase4_lines: list[str] = []
_phase4_lines.append("<!-- §9 sampling judgments -->\n")
for _label, _frame in [
    ("§9.1 human_review", _hr_sample),
    ("§9.2 review_floor_band", _floor_sample),
    ("§9.2 auto_merge_band", _auto_sample),
]:
    if _frame is None or _frame.empty:
        continue
    _phase4_lines.append(f"\n#### {_label} judgments\n")
    for _i, _row in _frame.iterrows():
        _phase4_lines.append(
            f"- PATID_A={_row['PATID_A']}, PATID_B={_row['PATID_B']}, "
            f"score={_row.get('match_probability', float('nan')):.4f}, "
            f"tier=`{_row['classification_tier']}`  \n"
            f"  - **Reviewer verdict:** [ true_match | not_match | unsure ]  \n"
            f"  - **Reviewer notes:**  \n"
        )

print("".join(_phase4_lines))


## 11. Three-way FS head-to-head — baseline vs enhanced vs enhanced_2

§10 compared the *baseline FS alone* against the *combined system* (deterministic rules + enhanced FS). §11 widens that comparison to **three FS variants**, each as the Stage-4 model inside the same combined system:

| Model | Training | Tunables | Special features |
|---|---|---|---|
| **baseline** | EM (3 sessions) | `0.90 / 0.50` (am / floor) | the original FS Splink build |
| **enhanced** | EM + manual priors (Phase E1 deterministic vetoes) | `0.95 / 0.40` | adds `Household_discount`, JW<0.5 mismatch levels, deterministic vetoes |
| **enhanced_2** | supervised m from synthetic positives | `0.95 / 0.40` | no vetoes (replaced by upstream `classify_non_matches` from Phase E2-5b); adds Middle name, Sex_positive, Phonetic-name, ZIP-base comparisons |

Each model is evaluated under the same Stage-3 pre-filter (the contradiction-counting `classify_non_matches` shipped on `develop`). The figures below answer four questions:

1. **Where does each model land on the 42 reviewer labels?** (Figure 1 — directional precision/recall.)
2. **Which model recovers known positives best?** (Figure 2 — silver-label recall, statistically powered.)
3. **How does each model spread its scores across the ambiguous review pool?** (Figure 3 — distributional separation.)
4. **Did Stage 3's contradiction-filter do what we thought it did?** (Figure 4 — pairs that Stage 3 routed to `reject` would have scored high in Stage 4.)

> §10 reads inputs from in-memory `df_scored` (baseline) + `data/matches/` + `data/matches_model/`. §11 reads the durable cross-model contract: `models/outputs/<model_name>__<data-version>.parquet` (the 5-col `eval_schema`). This is the apples-to-apples lane both runners write to.

_(§10 was removed in the E3-5 notebook trim; the head-to-head moved here as §11.)_


### 11.0 Setup — resolve artifacts, parse labels, build the joins

Loads the three eval-schema parquets (baseline / enhanced / enhanced_2), the enhanced_2 full-pool parquet, the Stage-3 review/reject frames, and any silver-label parquet from `data/silver_labels/`. Also defines the `df_labels_3way` frame the confusion-matrix figure consumes.


In [ ]:
# §11 SECTION MARKER — do not remove
# §11 setup — resolve artifacts, parse labels, attach per-model tiers.
import json as _json11
import re as _re11

from models.common.versioning import latest_versioned

# Helper: most-recent-by-mtime file resolver (used for run_id-stamped artifacts).
def _latest_by_mtime(dir_: Path, pattern: str) -> Path | None:
    if not dir_.exists():
        return None
    paths = sorted(dir_.glob(pattern), key=lambda p: p.stat().st_mtime)
    return paths[-1] if paths else None

OUTPUTS_DIR = PROJECT_ROOT / "models" / "outputs"
NON_MATCHES_DIR = PROJECT_ROOT / "data" / "non_matches"
REJECTS_DIR = PROJECT_ROOT / "data" / "rejects"
MATCHES_DIR_11 = PROJECT_ROOT / "data" / "matches"
SILVER_DIR = PROJECT_ROOT / "data" / "silver_labels"
ARTIFACTS_ENH2 = PROJECT_ROOT / "models" / "artifacts" / "fs_splink_enhanced_2"


def _resolve_eval(model_name: str, full_pool: bool = False) -> Path:
    # Only enhanced_2 has separate `_full_pool` vs non_matches variants. Baseline
    # and enhanced runners default to scoring the full candidate pool and write a
    # single output per data-version (no suffix). For those two models, both
    # full_pool=True and full_pool=False resolve to the same file — the cell then
    # filters to review-tier via an inner-join when a non_matches view is wanted.
    has_pool_variants = (model_name == "fs_splink_enhanced_2")
    if has_pool_variants:
        cands = sorted(OUTPUTS_DIR.glob(f"{model_name}__*.parquet"))
        cands = [c for c in cands
                 if c.stem.endswith("_full_pool") == full_pool
                 and "synthetic" not in c.stem]
    else:
        cands = sorted(OUTPUTS_DIR.glob(f"{model_name}__*.parquet"))
        cands = [c for c in cands
                 if not c.stem.endswith("_full_pool")
                 and "synthetic" not in c.stem]
    if not cands:
        raise FileNotFoundError(
            f"No eval-schema parquet for {model_name} (full_pool={full_pool}). "
            f"Looked under {OUTPUTS_DIR}."
        )
    # Most recent mtime wins (handles multiple data-version tags).
    return max(cands, key=lambda p: p.stat().st_mtime)


# Eval-schema parquets (5 cols: PATID_A, PATID_B, model_name, score, predicted_tier)
path_bl_full = _resolve_eval("fs_splink_baseline", full_pool=True)
path_enh_full = _resolve_eval("fs_splink_enhanced", full_pool=True)
path_enh2_nm = _resolve_eval("fs_splink_enhanced_2", full_pool=False)
path_enh2_full = _resolve_eval("fs_splink_enhanced_2", full_pool=True)

print(f"baseline  (full pool) : {path_bl_full.name}")
print(f"enhanced  (full pool) : {path_enh_full.name}")
print(f"enhanced_2 (non_matches): {path_enh2_nm.name}")
print(f"enhanced_2 (full pool) : {path_enh2_full.name}")

df_bl = pd.read_parquet(path_bl_full)
df_enh = pd.read_parquet(path_enh_full)
df_enh2 = pd.read_parquet(path_enh2_nm)
df_enh2_full = pd.read_parquet(path_enh2_full)

# Stage-3 frames (review = downstream non_matches; reject = dropped; matches = rule-confirmed)
nm_path = _latest_by_mtime(NON_MATCHES_DIR, "non_matches_*.parquet")
rj_path = _latest_by_mtime(REJECTS_DIR, "rejects_*.parquet")
mt_path = _latest_by_mtime(MATCHES_DIR_11, "matches_*.parquet")
print(f"non_matches (review): {nm_path}")
print(f"rejects             : {rj_path}")
print(f"matches (rule confirm): {mt_path}")

df_review = pd.read_parquet(nm_path)[["PATID_A", "PATID_B"]] if nm_path else pd.DataFrame(columns=["PATID_A","PATID_B"])
df_reject = pd.read_parquet(rj_path)[["PATID_A", "PATID_B"]] if rj_path else pd.DataFrame(columns=["PATID_A","PATID_B"])
df_matches11 = pd.read_parquet(mt_path)[["PATID_A","PATID_B"]] if mt_path else pd.DataFrame(columns=["PATID_A","PATID_B"])

# ── Silver labels (positives-only CSV, may be absent locally) ────────────────
df_silver = None
if SILVER_DIR.exists():
    _silver_paths = sorted(SILVER_DIR.glob("*.csv"))
    if _silver_paths:
        # Take the largest one — usually a single combined positives file
        _sp = max(_silver_paths, key=lambda p: p.stat().st_size)
        # dtype=str preserves PATID leading zeros (mirrors parquet roundtrip behaviour)
        df_silver = pd.read_csv(_sp, dtype=str)
        # Normalize column case if needed
        if "PATID_A" not in df_silver.columns:
            df_silver = df_silver.rename(columns={c: c.upper() for c in df_silver.columns if c.lower() in {"patid_a","patid_b"}})
        df_silver = df_silver[["PATID_A","PATID_B"]].drop_duplicates().reset_index(drop=True)
        print(f"silver labels      : {_sp.name}  (n={len(df_silver):,})")
    else:
        print("silver labels      : (no *.csv files in data/silver_labels/)")
else:
    print("silver labels      : (data/silver_labels/ absent — skipping recall figure)")

# ── Tier-column normalizer (eval_schema uses `predicted_tier`) ───────────────
def _norm_eval(df: pd.DataFrame, model: str) -> pd.DataFrame:
    out = df.rename(columns={
        "predicted_tier": f"tier_{model}",
        "score":          f"score_{model}",
    })
    return out[["PATID_A", "PATID_B", f"tier_{model}", f"score_{model}"]]

bl_norm = _norm_eval(df_bl, "baseline")
enh_norm = _norm_eval(df_enh, "enhanced")
enh2_norm = _norm_eval(df_enh2, "enhanced_2")
enh2_full_norm = _norm_eval(df_enh2_full, "enhanced_2")  # for silver-label join

# ── 42-label parse — regex over the "Reviewer judgments" markdown cell ─────
_label_re_11 = _re11.compile(
    r"PATID_A=<([0-9A-Fa-f]+)>\s*PATID_B=<([0-9A-Fa-f]+)>[^\n]*?score=<([0-9.]+)>"
    r"[^v]*?verdict:\s*<([a-z\- ]+)>",
    _re11.IGNORECASE,
)
_nb_path = Path("fellegi_sunter_validation.ipynb")
_nb = _json11.loads(_nb_path.read_text())
_labels_md = ""
for _c in _nb["cells"]:
    _src = "".join(_c["source"])
    if _c["cell_type"] == "markdown" and _src.lstrip().startswith("## Reviewer judgments"):
        _labels_md = _src
        break
_labels = [{
    "PATID_A": _m.group(1).upper(),
    "PATID_B": _m.group(2).upper(),
    "verdict": _m.group(4).strip().lower(),
} for _m in _label_re_11.finditer(_labels_md)]
df_labels_3way = pd.DataFrame(_labels).drop_duplicates(["PATID_A","PATID_B"]).reset_index(drop=True)
df_labels_3way["verdict_norm"] = df_labels_3way["verdict"].map(
    lambda v: {"same":"same","different":"different","unsure":"unsure"}.get(v, "unsure")
)
print(f"Parsed {len(df_labels_3way)} labeled pairs")

# ── Attach per-model COMBINED-SYSTEM tier ────────────────────────────────────
# Combined tier rules:
#   1. rule-confirmed (in df_matches11)              → auto_merge
#   2. else, tier from this model's non_matches-only score (filter full-pool to review only for fair comparison)
#   3. else (pair is rejected or never blocked)       → no_match
def _restrict_to_review(scored: pd.DataFrame, review: pd.DataFrame, tier_col: str) -> pd.DataFrame:
    if review.empty:
        return scored
    return scored.merge(review, on=["PATID_A","PATID_B"], how="inner")

bl_review = _restrict_to_review(bl_norm, df_review, "tier_baseline")
enh_review = _restrict_to_review(enh_norm, df_review, "tier_enhanced")
# enhanced_2 default IS non_matches already → use as-is
enh2_review = enh2_norm.copy()

# Build df_labels_3way with three combined tiers
df_rules_flag = df_matches11.copy()
df_rules_flag["rule_confirmed"] = True
df_labels_3way = df_labels_3way.merge(df_rules_flag, on=["PATID_A","PATID_B"], how="left")
df_labels_3way["rule_confirmed"] = df_labels_3way["rule_confirmed"].fillna(False)

for label, frame in [("baseline", bl_review), ("enhanced", enh_review), ("enhanced_2", enh2_review)]:
    df_labels_3way = df_labels_3way.merge(frame, on=["PATID_A","PATID_B"], how="left")
    def _combine(row, lbl=label):
        if row["rule_confirmed"]:
            return "auto_merge"
        t = row.get(f"tier_{lbl}")
        if pd.notna(t):
            return t
        return "no_match"
    df_labels_3way[f"combined_{label}"] = df_labels_3way.apply(_combine, axis=1)

print(df_labels_3way[["verdict_norm","combined_baseline","combined_enhanced","combined_enhanced_2"]]
      .value_counts(dropna=False).head(10))


### 11.1 Confusion matrix — 3-panel on 42 reviewer labels

Each panel shows the **combined-system tier** (rules + that FS model) versus the reviewer verdict on the 42 labeled pairs. Rows are reviewer verdicts (`different` / `unsure` / `same`); columns are predicted tiers (`no_match` / `human_review` / `auto_merge`).

**Read direction:** for high precision you want the `different → auto_merge` cell empty; for high recall you want the `same → auto_merge` cell to dominate the `same` row.

⚠ 42 labels is a small sample. The CM tells you which model is *directionally* most calibrated — recall claims under this view are noisy. §11.2 covers recall under a higher-power view.


In [ ]:
# §11.1 — 3-panel confusion matrix.
_TIER_ORDER_11 = ["no_match", "human_review", "auto_merge"]
_VERDICT_ORDER_11 = ["different", "unsure", "same"]
_MODELS_11 = [("baseline", "Baseline FS  (0.90 / 0.50)"),
              ("enhanced", "Enhanced FS  (0.95 / 0.40)"),
              ("enhanced_2", "Enhanced_2 FS  (0.95 / 0.40)")]


def _cm(df: pd.DataFrame, tier_col: str) -> pd.DataFrame:
    return pd.crosstab(df["verdict_norm"], df[tier_col]).reindex(
        index=_VERDICT_ORDER_11, columns=_TIER_ORDER_11, fill_value=0
    )


cms = [(label, title, _cm(df_labels_3way, f"combined_{label}")) for label, title in _MODELS_11]
_vmax = max(cm.values.max() for _, _, cm in cms)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
for ax, (label, title, cm) in zip(axes, cms):
    ax.imshow(cm.values, cmap="Greens", vmin=0, vmax=max(_vmax, 1))
    ax.set_xticks(range(len(_TIER_ORDER_11)))
    ax.set_xticklabels(_TIER_ORDER_11, rotation=20, ha="right")
    ax.set_yticks(range(len(_VERDICT_ORDER_11)))
    ax.set_yticklabels(_VERDICT_ORDER_11)
    ax.set_xlabel("Predicted tier")
    ax.set_ylabel("Reviewer verdict")
    ax.set_title(title)
    for (i, j), v in np.ndenumerate(cm.values):
        ax.text(j, i, str(int(v)), ha="center", va="center",
                color="white" if v > _vmax / 2 else "#222")
    ax.grid(False)

fig.suptitle(f"3-way confusion matrix — 42 labeled pairs  ·  {VERSION_TAG}",
             fontsize=13, fontweight="bold", y=1.06)
out_path = FIGURES_DIR / f"confusion_matrix_3way__{VERSION_TAG}.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")


# ── Quantitative summary ─────────────────────────────────────────────────────
def _summarise_3way(label: str, cm: pd.DataFrame) -> None:
    n = int(cm.values.sum())
    am_diff = int(cm.loc["different", "auto_merge"])
    am_same = int(cm.loc["same", "auto_merge"])
    am_total = int(cm["auto_merge"].sum())
    same_total = int(cm.loc["same"].sum())
    am_prec = (am_same / am_total) if am_total else float("nan")
    same_rec = ((cm.loc["same","auto_merge"] + cm.loc["same","human_review"]) / same_total) if same_total else float("nan")
    print(f"{label:11s}: n={n} | AM precision = {am_prec:.0%} ({am_same}/{am_total}) | "
          f"same-recall to AM∪HR = {same_rec:.0%} | false-merges (diff→AM) = {am_diff}")


for label, _, cm in cms:
    _summarise_3way(label, cm)


### 11.2 Silver-label recall by tier — 3 models × 3 tiers grouped bar

Silver labels are **positives-only** (Stage-3-confirmed pairs from the `data/silver_labels/` validation set, ~99% adjudicator precision). They give a statistically-powered view of one direction: **of all known positives, what fraction does each model promote to auto_merge / hold for human review / drop to no_match?**

This is not a confusion matrix — there are no negatives to test against — so don't try to read precision off it. But the recall direction is reliable here in a way the 42-label CM is not.

Silver labels are filtered out of `non_matches` by Stage 3 (because they were confirmed), so we read them off the **full-pool** scoring parquets. Pairs that aren't in the full-pool output are assumed `no_match` (e.g., never blocked).


In [ ]:
# §11.2 — silver-label recall@tier (grouped bar).
if df_silver is None or df_silver.empty:
    print("§11.2 SKIPPED — no silver labels available locally. Re-run on the VM "
          "where `data/silver_labels/*.parquet` exists.")
else:
    def _silver_tier(model_norm_full: pd.DataFrame, model_label: str) -> pd.Series:
        m = df_silver.merge(
            model_norm_full[["PATID_A","PATID_B", f"tier_{model_label}"]],
            on=["PATID_A","PATID_B"], how="left",
        )
        return m[f"tier_{model_label}"].fillna("no_match")

    _silver_tiers = {
        "baseline":   _silver_tier(bl_norm, "baseline"),
        "enhanced":   _silver_tier(enh_norm, "enhanced"),
        "enhanced_2": _silver_tier(enh2_full_norm, "enhanced_2"),  # full-pool variant
    }

    _grouped = pd.DataFrame({
        m: t.value_counts().reindex(_TIER_ORDER_11, fill_value=0)
        for m, t in _silver_tiers.items()
    }).T
    _grouped_pct = _grouped.div(_grouped.sum(axis=1), axis=0) * 100.0

    fig, ax = plt.subplots(figsize=(11, 4.6))
    _x = np.arange(len(_grouped_pct.index))
    _width = 0.26
    for i, tier in enumerate(_TIER_ORDER_11):
        ax.bar(_x + (i - 1) * _width, _grouped_pct[tier].values, _width,
               color=TIER_COLORS[tier], edgecolor=TIER_EDGE[tier],
               label=tier, linewidth=1.0)
        for xi, v in zip(_x + (i - 1) * _width, _grouped_pct[tier].values):
            ax.text(xi, v + 1, f"{v:.0f}%", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(_x)
    ax.set_xticklabels(_grouped_pct.index)
    ax.set_ylabel("% of silver-label positives")
    ax.set_ylim(0, 105)
    ax.set_title(f"Silver-label recall@tier — n={len(df_silver):,} known-positive pairs  ·  {VERSION_TAG}",
                 fontsize=13, fontweight="bold")
    ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.08))
    ax.grid(axis="y", alpha=0.25)

    out_path = FIGURES_DIR / f"silver_label_recall_by_tier__{VERSION_TAG}.png"
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
    print()
    print("Silver-label tier counts (rows = model, cols = tier):")
    print(_grouped)


### 11.3 Score distribution — 3-panel histograms on the post-rules `non_matches` pool

Each panel is the score histogram a model assigns to the *review*-tier pool (post-rules `non_matches`). Log y-axis surfaces the tail. The shaded vertical bands are that model's own `[0, review_floor)` / `[review_floor, auto_merge_threshold)` / `[auto_merge_threshold, 1]` regions.

**Reading guide:** a well-calibrated model concentrates mass at the floor (most ambiguous pairs are true negatives) and produces a clean spike near 1.0 only for the genuine merges. A *uniform spread* across `[floor, auto_merge]` is the warning sign — undifferentiated middle scores.


In [ ]:
# §11.3 — 3-panel score distribution on the post-rules non_matches pool.
# Filter each model's full-pool eval-schema to review-tier pairs for fair comparison.
_BL_FLOOR, _BL_AM = 0.50, 0.90
_EN_FLOOR, _EN_AM = 0.40, 0.95
_E2_FLOOR, _E2_AM = 0.40, 0.95

_panels = [
    ("baseline",   _restrict_to_review(bl_norm,  df_review, "tier_baseline"),   "score_baseline",   _BL_FLOOR, _BL_AM, "Baseline FS"),
    ("enhanced",   _restrict_to_review(enh_norm, df_review, "tier_enhanced"),   "score_enhanced",   _EN_FLOOR, _EN_AM, "Enhanced FS"),
    ("enhanced_2", enh2_norm,                                                    "score_enhanced_2", _E2_FLOOR, _E2_AM, "Enhanced_2 FS"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
_bins = np.linspace(0, 1, 51)

for ax, (label, frame, score_col, floor, am, title) in zip(axes, _panels):
    if frame.empty or score_col not in frame.columns:
        ax.text(0.5, 0.5, f"{label}: no data", ha="center", va="center")
        ax.set_title(title)
        continue
    s = frame[score_col].dropna().to_numpy()
    ax.axvspan(0.0, floor, color=TIER_COLORS["no_match"], alpha=0.18, lw=0)
    ax.axvspan(floor, am, color=TIER_COLORS["human_review"], alpha=0.22, lw=0)
    ax.axvspan(am, 1.0, color=TIER_COLORS["auto_merge"], alpha=0.22, lw=0)
    ax.hist(s, bins=_bins, color="#1F4E79", edgecolor="white", linewidth=0.4)
    ax.axvline(floor, color="#555", linestyle="--", linewidth=0.8)
    ax.axvline(am, color="#555", linestyle="--", linewidth=0.8)
    ax.set_yscale("log")
    ax.set_xlim(0, 1)
    ax.set_xlabel("score")
    ax.set_ylabel("count (log)")
    ax.set_title(f"{title}  ·  n={len(s):,}")
    ax.text(floor, ax.get_ylim()[1] * 0.6, f" floor={floor:.2f}", fontsize=8, color="#444")
    ax.text(am,    ax.get_ylim()[1] * 0.6, f" AM={am:.2f}",       fontsize=8, color="#444")

fig.suptitle(f"Score distribution — post-rules non_matches pool  ·  {VERSION_TAG}",
             fontsize=13, fontweight="bold", y=1.06)
out_path = FIGURES_DIR / f"score_distribution_3way__{VERSION_TAG}.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")


### 11.4 Reject vs review — validates the upstream `classify_non_matches` filter

For every pair in the **enhanced_2 full-pool** scoring artifact, attach the Stage-3 decision (`reject` / `review` / `confirmed`) and overlay their score histograms.

**What this shows:** if the upstream contradiction filter is doing its job, the `reject`-tier pairs should pile up at high FS scores — i.e., FS *would have* promoted many of them to `auto_merge` if Stage 3 hadn't dropped them. A `reject` distribution heavily right-shifted is the validation that the upstream filter prevented a real FP burst. A `reject` distribution overlapping `review` says Stage 3 dropped pairs that Stage 4 would also have handled correctly — i.e., the filter is over-aggressive.


In [ ]:
# §11.4 — reject vs review score-distribution overlay (enhanced_2 full-pool).
_full = df_enh2_full.copy()
# Tag each pair by its Stage-3 decision.
_full = _full.merge(df_matches11.assign(_dec="confirmed"), on=["PATID_A","PATID_B"], how="left")
_full = _full.merge(df_review.assign(_rev=True), on=["PATID_A","PATID_B"], how="left")
_full = _full.merge(df_reject.assign(_rej=True), on=["PATID_A","PATID_B"], how="left")


def _stage3_decision(row: pd.Series) -> str:
    if row.get("_dec") == "confirmed":
        return "confirmed"
    if row.get("_rej") is True:
        return "reject"
    if row.get("_rev") is True:
        return "review"
    return "unblocked"

_full["stage3"] = _full.apply(_stage3_decision, axis=1)
print("Stage-3 decision counts in enhanced_2 full-pool:")
print(_full["stage3"].value_counts())

fig, ax = plt.subplots(figsize=(11, 4.6))
_bins = np.linspace(0, 1, 51)
_palette = {"confirmed": "#2D7F4B", "review": "#A14B2A", "reject": "#B2182B"}
for decision in ["confirmed", "review", "reject"]:
    s = _full.loc[_full["stage3"] == decision, "score"].dropna().to_numpy()
    if len(s) == 0:
        continue
    ax.hist(s, bins=_bins, density=True, histtype="step", linewidth=2.0,
            color=_palette[decision], label=f"{decision}  (n={len(s):,})")

ax.axvline(_E2_FLOOR, color="#555", linestyle="--", linewidth=0.8)
ax.axvline(_E2_AM, color="#555", linestyle="--", linewidth=0.8)
ax.text(_E2_FLOOR, ax.get_ylim()[1] * 0.95, f" floor={_E2_FLOOR}", fontsize=9, color="#444")
ax.text(_E2_AM,    ax.get_ylim()[1] * 0.95, f" AM={_E2_AM}",       fontsize=9, color="#444")
ax.set_xlabel("enhanced_2 score")
ax.set_ylabel("density")
ax.set_xlim(0, 1)
ax.legend(loc="upper center")
ax.set_title(f"Stage-3 decision vs enhanced_2 score — full candidate pool  ·  {VERSION_TAG}",
             fontsize=13, fontweight="bold")

out_path = FIGURES_DIR / f"stage3_decision_vs_enhanced_2_score__{VERSION_TAG}.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## 12. Enhanced_3 — full-cohort FS evaluation (silver-label trained)

enhanced_3 is the deliberately-simple FS variant: seven two-level (Exact / All-other) comparisons, m trained **supervised from the real-cohort silver labels**, lambda seeded from the deterministic rules, u from random sampling. This section trains it and runs **all Alliance data through the FS model directly** — the *full candidate pool*, **bypassing** the deterministic-rules stage — then renders the full evaluation suite.

| Aspect | enhanced_3 |
|---|---|
| Comparisons | FirstNM, LastNM, BirthDT, SSN, Email, Phones, Address — each null / exact / all-other |
| m training | `estimate_m_from_pairwise_labels` on the silver-label TRAIN split |
| lambda | `estimate_probability_two_random_records_match` (deterministic PRIOR_RULES) |
| u | `estimate_u_using_random_sampling` |
| Tiers | `0.95 / 0.40` (auto_merge / review_floor) |

> **VM-only.** Needs `data/silver_labels/silver_labels_v1_2026_06_21.csv` (gitignored PHI) and the full candidate-pairs parquet. Off-VM the setup cell skips and the rest of §12 no-ops.


### 12.0 Setup — train enhanced_3, score the full candidate pool, score the test split

Splits the silver labels (stratified, seed 42) into TRAIN (→ m) and TEST (held out). Trains `FSEnhanced3` and scores the entire candidate pool with `return_linker=True` so the trained Splink linker is available for the native charts below. Test-split metrics come from joining the scored full-pool output to the held-out labels (the silver pairs are present in the full pool by construction).


In [ ]:
# §12 SECTION MARKER — do not remove
# §12.0 setup — train enhanced_3 + score full candidate pool + test split.
import numpy as _np12

from models.common.versioning import latest_versioned as _latest_versioned_12
from models.experiments.fs_splink_enhanced_3.fs_enhanced_3 import FSEnhanced3

# Tier palette / figure conventions are defined in §8.0; redefine defensively.
try:
    _ = TIER_COLORS  # type: ignore[name-defined]
except NameError:
    TIER_COLORS = {"no_match": "#B5B5B5", "human_review": "#F0BE7E", "auto_merge": "#88B888"}
    TIER_EDGE = {"no_match": "#7F7F7F", "human_review": "#C68A3F", "auto_merge": "#4F8A4F"}
    TIER_ORDER_LOW_TO_HIGH = ["no_match", "human_review", "auto_merge"]

E3_AM, E3_FLOOR = 0.95, 0.40
SILVER_PATH_12 = PROJECT_ROOT / "data" / "silver_labels" / "silver_labels_v1_2026_06_21.csv"

# Resolve the FULL candidate-pairs parquet (VM convention: src/preprocessing/outputs/blocking;
# also try data/blocking). enhanced_3 scores this directly — no deterministic-rules stage.
def _resolve_full_candidate_pairs_12() -> Path | None:
    for _d in (
        PROJECT_ROOT / "src" / "preprocessing" / "outputs" / "blocking",
        PROJECT_ROOT / "data" / "blocking",
        PROJECT_ROOT / "src" / "features" / "outputs" / "blocking",
    ):
        if not _d.is_dir():
            continue
        try:
            return _latest_versioned_12(_d, "candidate_pairs_v*_*.parquet")
        except (FileNotFoundError, Exception):
            _g = sorted(_d.glob("candidate_pairs_*.parquet"))
            if _g:
                return _g[-1]
    return None

E3_READY = False
if not SILVER_PATH_12.exists():
    print(f"§12 SKIPPED — silver labels absent ({SILVER_PATH_12}). VM-only section.")
else:
    _cp_path = _resolve_full_candidate_pairs_12()
    if _cp_path is None:
        print("§12 SKIPPED — no candidate_pairs parquet found. Generate the full "
              "blocking output first (run_blocking.py).")
    else:
        print(f"silver labels   : {SILVER_PATH_12.name}")
        print(f"candidate pairs : {_cp_path.name}")
        # cleaned_path / df_clean are resolved/loaded earlier in the notebook (§2-3).
        _silver = pd.read_csv(SILVER_PATH_12, dtype={"PATID_A": str, "PATID_B": str})
        # Silver labels: column `silver_label` with boolean True/False. Normalize
        # to an int `label` column (1/0) for the joins + groupby below.
        _src_col = "silver_label" if "silver_label" in _silver.columns else "label"
        _silver["label"] = _silver[_src_col].map(
            {True: 1, False: 0, "True": 1, "False": 0, 1: 1, 0: 0}
        ).astype(int)

        # Stratified 80/20 split (seed 42) — matches run_real_enhanced_3 defaults.
        _test = pd.concat([g.sample(frac=0.2, random_state=42)
                           for _, g in _silver.groupby("label")])
        _train = _silver.drop(_test.index).reset_index(drop=True)
        _test = _test.reset_index(drop=True)
        print(f"silver split    : train {len(_train)} (pos={int((_train.label==1).sum())}) "
              f"/ test {len(_test)} (pos={int((_test.label==1).sum())})")

        _cand = pd.read_parquet(_cp_path)
        print(f"Scoring full candidate pool: {len(_cand):,} pairs ...")

        _e3_model = FSEnhanced3(labels_df=_train, include_address=True, u_max_pairs=1e6)
        try:
            e3_classified, e3_linker = _e3_model.run(
                _cand, df_clean, full_output=True, return_linker=True,
            )
        except RuntimeError as _exc:
            print(f"Retrying with u_max_pairs=1e4 ({_exc})")
            _e3_model = FSEnhanced3(labels_df=_train, include_address=True, u_max_pairs=1e4)
            e3_classified, e3_linker = _e3_model.run(
                _cand, df_clean, full_output=True, return_linker=True,
            )

        # Held-out test metrics: canonicalize test pairs, inner-join to scored pool.
        _tc = _test.copy()
        _a = _tc[["PATID_A", "PATID_B"]].min(axis=1)
        _b = _tc[["PATID_A", "PATID_B"]].max(axis=1)
        _tc["PATID_A"], _tc["PATID_B"] = _a, _b
        e3_test = e3_classified.merge(
            _tc[["PATID_A", "PATID_B", "label"]], on=["PATID_A", "PATID_B"], how="inner",
        )
        print(f"test pairs scored in full pool: {len(e3_test)}/{len(_test)}")

        _tiers = e3_classified["classification_tier"].value_counts().to_dict()
        print("Full-pool tier breakdown:", {k: int(v) for k, v in _tiers.items()})
        E3_READY = True


def _save_altair_12(chart, name: str):
    """Save a Splink/Altair chart as HTML (always) + PNG (if vl-convert present)."""
    base = FIGURES_DIR / f"{name}__{VERSION_TAG}"
    try:
        chart.save(str(base) + ".html")
        print(f"Saved: {base.name}.html")
    except Exception as _e:
        print(f"  (HTML save failed for {name}: {_e})")
    try:
        chart.save(str(base) + ".png")
        print(f"Saved: {base.name}.png")
    except Exception:
        print(f"  (PNG save skipped for {name} — install vl-convert-python to enable)")
    return chart


### 12.1 Performance — waterfall, confusion matrix, score histogram

1. **Waterfall chart** (Splink) — decomposes a handful of scored pairs into the per-feature match-weight contribution, so you can see *which* of the seven fields drove each score.
2. **Confusion matrix** (test split) — tier vs silver label on the held-out pairs (rows = label, cols = tier).
3. **Score histogram** — full-pool score distribution, stacked + coloured by tier, with the `0.40 / 0.95` threshold bands.


In [ ]:
# §12.1 — waterfall + confusion matrix + score histogram.
if not E3_READY:
    print("§12.1 SKIPPED — setup did not complete.")
else:
    # (1) Waterfall — a few representative scored records from the trained linker.
    try:
        _wf_records = e3_linker.inference.predict().as_record_dict(limit=12)
        _wf = e3_linker.visualisations.waterfall_chart(_wf_records, filter_nulls=False)
        _save_altair_12(_wf, "enhanced_3_waterfall")
        display(_wf)
    except Exception as _e:
        print(f"§12.1 waterfall skipped: {_e}")

    # (2) Confusion matrix on the held-out test split.
    _TIER_ORDER = ["no_match", "human_review", "auto_merge"]
    _cm = (pd.crosstab(e3_test["label"], e3_test["classification_tier"])
             .reindex(index=[0, 1], columns=_TIER_ORDER, fill_value=0))
    fig, ax = plt.subplots(figsize=(6.2, 3.6), constrained_layout=True)
    ax.imshow(_cm.values, cmap="Greens", vmin=0, vmax=max(_cm.values.max(), 1))
    ax.set_xticks(range(3)); ax.set_xticklabels(_TIER_ORDER, rotation=20, ha="right")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["different (0)", "same (1)"])
    ax.set_xlabel("Predicted tier"); ax.set_ylabel("Silver label")
    for (i, j), v in _np12.ndenumerate(_cm.values):
        ax.text(j, i, str(int(v)), ha="center", va="center",
                color="white" if v > _cm.values.max() / 2 else "#222")
    ax.set_title(f"enhanced_3 — test confusion  ·  {VERSION_TAG}", fontweight="bold")
    ax.grid(False)
    _out = FIGURES_DIR / f"enhanced_3_confusion__{VERSION_TAG}.png"
    plt.savefig(_out); plt.show(); print(f"Saved: {_out.name}")

    # Precision / recall at auto_merge (test split).
    _tp = int(((e3_test.label == 1) & (e3_test.classification_tier == "auto_merge")).sum())
    _fp = int(((e3_test.label == 0) & (e3_test.classification_tier == "auto_merge")).sum())
    _fn = int(((e3_test.label == 1) & (e3_test.classification_tier != "auto_merge")).sum())
    _prec = _tp / (_tp + _fp) if (_tp + _fp) else float("nan")
    _rec = _tp / (_tp + _fn) if (_tp + _fn) else float("nan")
    print(f"auto_merge — precision={_prec:.1%}  recall={_rec:.1%}  (tp={_tp} fp={_fp} fn={_fn})")

    # (3) Full-pool score histogram, stacked by tier, with threshold bands.
    _bins = _np12.linspace(0, 1, 51)
    _centers = (_bins[:-1] + _bins[1:]) / 2
    fig, ax = plt.subplots(figsize=(11, 4.6))
    _cum = _np12.zeros(len(_centers))
    for _t in TIER_ORDER_LOW_TO_HIGH:
        _s = e3_classified.loc[e3_classified["classification_tier"] == _t, "match_probability"]
        _h, _ = _np12.histogram(_s.to_numpy(), bins=_bins)
        ax.bar(_centers, _h, width=(_bins[1] - _bins[0]) * 0.95, bottom=_cum,
               color=TIER_COLORS[_t], edgecolor=TIER_EDGE[_t], linewidth=0.4,
               label=f"{_t}  ({int(_h.sum()):,})")
        _cum = _cum + _h
    ax.axvline(E3_FLOOR, color="#555", ls="--", lw=0.8)
    ax.axvline(E3_AM, color="#555", ls="--", lw=0.8)
    ax.set_yscale("log"); ax.set_xlim(0, 1)
    ax.set_xlabel("match probability"); ax.set_ylabel("count (log)")
    ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.12))
    ax.set_title(f"enhanced_3 — full-pool score distribution  ·  {VERSION_TAG}", fontweight="bold")
    _out = FIGURES_DIR / f"enhanced_3_score_histogram__{VERSION_TAG}.png"
    plt.savefig(_out, bbox_inches="tight"); plt.show(); print(f"Saved: {_out.name}")


### 12.2 Model diagnostics — Splink parameter charts

Splink's native diagnostic charts on the trained linker:

- **match_weights_chart** — the learned weight (log2 Bayes factor) for every comparison level.
- **m_u_parameters_chart** — the underlying m and u probabilities each weight is derived from.
- **parameter_estimate_comparisons_chart** — compares estimates across training passes (here: the supervised m vs the random-sampling u).
- **tf_adjustment_chart** — per-value term-frequency adjustment for the high-cardinality identity fields (FirstNM, LastNM, Email).


In [ ]:
# §12.2 — Splink parameter diagnostic charts.
if not E3_READY:
    print("§12.2 SKIPPED — setup did not complete.")
else:
    for _name, _fn in [
        ("enhanced_3_match_weights", lambda: e3_linker.visualisations.match_weights_chart()),
        ("enhanced_3_m_u_parameters", lambda: e3_linker.visualisations.m_u_parameters_chart()),
        ("enhanced_3_parameter_estimates",
         lambda: e3_linker.visualisations.parameter_estimate_comparisons_chart()),
    ]:
        try:
            _c = _fn()
            _save_altair_12(_c, _name)
            display(_c)
        except Exception as _e:
            print(f"§12.2 {_name} skipped: {_e}")

    # tf_adjustment_chart is per-column — only the TF-enabled identity fields.
    for _col in ["FirstNM_clean", "LastNM_clean", "Email_clean"]:
        try:
            _c = e3_linker.visualisations.tf_adjustment_chart(_col)
            _save_altair_12(_c, f"enhanced_3_tf_adjustment_{_col}")
            display(_c)
        except Exception as _e:
            print(f"§12.2 tf_adjustment[{_col}] skipped: {_e}")


### 12.3 Edge / link evaluation — accuracy + threshold-selection from the test labels

Registers the held-out **test split** as a Splink labels table and runs `linker.evaluation.accuracy_analysis_from_labels_table`:

- **accuracy chart** (`output_type="accuracy"`) — precision / recall / F1 etc. as a function of the match-weight threshold.
- **threshold-selection tool** (`output_type="threshold_selection"`) — the interactive tool for picking the operating threshold from labelled truth.

The labels table uses Splink's expected schema (`unique_id_l`, `unique_id_r`, `clerical_match_score`); `clerical_match_score` is the silver label (1.0 / 0.0).


In [ ]:
# §12.3 — accuracy + threshold-selection from the test labels table.
if not E3_READY:
    print("§12.3 SKIPPED — setup did not complete.")
else:
    try:
        # Build a Splink labels table from the held-out test split.
        _lab = _tc[["PATID_A", "PATID_B", "label"]].copy()
        _labels_tbl = pd.DataFrame({
            "unique_id_l": _lab["PATID_A"].astype(str),
            "unique_id_r": _lab["PATID_B"].astype(str),
            "source_dataset_l": "__splink__input_table_0",
            "source_dataset_r": "__splink__input_table_0",
            "clerical_match_score": _lab["label"].astype(float),
        })
        _registered = e3_linker.table_management.register_labels_table(
            _labels_tbl, overwrite=True,
        )
        for _name, _otype in [
            ("enhanced_3_accuracy", "accuracy"),
            ("enhanced_3_threshold_selection", "threshold_selection"),
        ]:
            try:
                _c = e3_linker.evaluation.accuracy_analysis_from_labels_table(
                    _registered, output_type=_otype,
                )
                _save_altair_12(_c, _name)
                display(_c)
            except Exception as _e:
                print(f"§12.3 {_name} skipped: {_e}")
    except Exception as _e:
        print(f"§12.3 labels-table registration failed: {_e}")
        print("  Fallback: see the §12.1 confusion matrix + precision/recall print.")


### 12.4 Derived match-weight table (m / u / log2 Bayes factor)

The headline interpretability artifact: one row per comparison level with the trained m, u, and the log2 Bayes factor (the match weight). Saved as CSV for `docs/Fellegi-Sunter-Enhanced_3.md` and printed as a markdown table to paste into the build guide.


In [ ]:
# §12.4 — derived match-weight table.
if not E3_READY:
    print("§12.4 SKIPPED — setup did not complete.")
else:
    _settings = e3_linker.misc.save_model_to_json()
    _rows = []
    for _comp in _settings.get("comparisons", []):
        _cname = _comp.get("output_column_name", "?")
        for _lvl in _comp.get("comparison_levels", []):
            if _lvl.get("is_null_level"):
                continue
            _m, _u = _lvl.get("m_probability"), _lvl.get("u_probability")
            _bf = (_np12.log2(_m / _u) if (_m and _u) else None)
            _rows.append({
                "comparison": _cname,
                "level": _lvl.get("label_for_charts", _lvl.get("sql_condition", "?")),
                "m": _m, "u": _u,
                "log2_bayes_factor": (round(_bf, 3) if _bf is not None else None),
            })
    _mw = pd.DataFrame(_rows)
    _out_csv = FIGURES_DIR / f"enhanced_3_match_weights__{VERSION_TAG}.csv"
    _mw.to_csv(_out_csv, index=False)
    print(f"Saved: {_out_csv.name}")
    print()
    # Markdown table for the doc (falls back to plain print if tabulate absent).
    try:
        print(_mw.to_markdown(index=False))
    except Exception as _e:
        print(f"(markdown render needs `tabulate`: {_e})")
        print(_mw.to_string(index=False))


## Reviewer judgments

 - Pair: PATID_A=<682F05B6FC0B423E4BAF2E65973EE907> PATID_B=<E6B86E52BF368B579B5D7C1AF47F3AF6> | section=§9.1 | score=<0.7406>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + exact address + same phone set>
    notes: <no additional notes to report>


- Pair: PATID_A=<AABECF0FB6679FD22AB032966012FFEB> PATID_B=<F615484F791DDE244D9F07BBEEA32761> | section=§9.1 | score=<0.7294>
    verdict: <different>
    confidence: <medium>
    pattern: <roomate-same-address>
    address: <same household>
    drivers: <one line>
    notes: <The address is technically exact however one patient record has spaces in between the street information whereas the other one does not>

- Pair: PATID_A=<70157C3E98BE59DEAE4A984BAB7559A9> PATID_B=<9F2BE9307A372B4758DAD8383CA30080> | section=§9.1 | score=<0.5677>
    verdict: <different>
    confidence: <high>
    pattern: <namesake-same-name>
    address: <different>
    drivers: <different first name, same last name, same DOB, different address, different phone set>
    notes: <The two records only matched on last name and DOB but had completely different addresses, zipcodes, phone numbers, and first names which was odd>


- Pair: PATID_A=<1CF523B0FFB944EDA2F6CAE741C8B72E> PATID_B=<97732CBC73F85505A764A5765D969913> | section=§9.1 | score=<0.8830>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + exact address + same email + different DOB>
    notes: <no additional notes to report>


- Pair: PATID_A=<378FA81D4D91666C9278E28505A38BCC> PATID_B=<C6A30F008554A9D97B9C1FB1AC132825> | section=§9.1 | score=<0.7800>
    verdict: <unsure>
    confidence: <low>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <other>
    notes: <The data is odd because both records match apart from the last name and date of birth. Originally I had thought it was the same person but just a name change due to marriage but that then the date of birth would have been the same so final verdice is unsure.>

- Pair: PATID_A=<7B2EF72A421AE6A58CA6366D41C1A1CA> PATID_B=<D90C5275BFFEB04D378E5182694623B8> | section=§9.1 | score=<0.6194>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <unusable>
    drivers: <same last name + same phone number>
    notes: <The address fields for one of the PATID_A are 'NA' so unclear if these records are from patients that live in the same household but just different individuals>

- Pair: PATID_A=<1EC8E1BEA622557873D992BF00D7C507> PATID_B=<9570F442CDA5997423B3B023755F14E3> | section=§9.1 | score=<0.6514>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<291CFBB3004F9A91F97ADED63845C58A> PATID_B=<D0AD271E455D6E7C23D748BAC472F949> | section=§9.1 | score=<0.8631>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<0949BC832B61AEB6F749F62D21C26812> PATID_B=<1D8E93DBB01A5497A0394F9DD90179F2> | section=§9.1 | score=<0.6500>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<081D634EF5668AE1DED0F305CE8383B3> PATID_B=<36DF6F5E0272DD9F757416F9694A28A8> | section=§9.1 | score=<0.7753>
    verdict: <unsure>
    confidence: <medium>
    pattern: <other>
    address: <exact>
    drivers: <different last name + different DOB + same first name + exact address + same phone number>
    notes: <the data is odd because the records match on first name but have different last name + different DOB but same exact address and phone number. It is also difficult to determine the verdict because PATID_A has 'NA' in Social security field.>

- Pair: PATID_A=<BC977805B1B9E6CF6661DB12F0BA3F5B> PATID_B=<CC812C5D2CCAD8BBFD55CC3626572AF9> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <medium>
    pattern: <roommate-same-household>
    address: <exact>
    drivers: <different first name + different last name + different DOB but exact address and phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<16E2E01F3491DA321A0A6080C91004D5> PATID_B=<C04DD69888E0B68D8B18400239714D00> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <same-household>
    drivers: <different first name, same-ish last name, same household, same phone number, same email >
    notes: <the last name for PATID_B has an additional name in it however there is enough overlap to suggest it is still the same last name shared by the two records>
   
- Pair: PATID_A=<266A55B6F2DA2F18D726E349E36E056F> PATID_B=<DFF9ADD06147B852E27EF51C22F02ECA> | section=§9.1 | score=<0.5290>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same last name + different first name + different DOB + different address + same phone number>
    notes: <The last name is shared between the two records and the same city + state + zipcode however the AL1 differs. This suggests potentially same family but live in different addresses now.>

- Pair: PATID_A=<450479ED367FC02CD538DF431FD722F4> PATID_B=<7D96EB724CFFDC20E010ED9A670A2E41> | section=§9.1 | score=<0.8607>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same last name + different first name + different DOB + different address + same phone number>
    notes: <The last name is shared between the two records and the same city + state + zipcode however the AL1 differs. This suggests potentially same family but live in different addresses now.>

- Pair: PATID_A=<28DECA1758F0E1CEE8374E33B366009E> PATID_B=<65AEA6D9C8DC70E66A9C53915A1B2AD6> | section=§9.1 | score=<0.5090>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<9B8F2F432876EC6D2805C79510FE7A8B> PATID_B=<F4644B8B7EDE7B3097568C1920170C1C> | section=§9.1 | score=<0.6425>
    verdict: <different>
    confidence: <medium>
    pattern: <roommate-same-address>
    address: <exact>
    drivers: <different first name + different last name + different DOB + exact address + same email + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<4F1228261B37E2663D85684B6F5F23BF> PATID_B=<8E6BF0201F5EDC75DB73C66299643901> | section=§9.1 | score=<0.8516>
    verdict: <different>
    confidence: <low>
    pattern: <other>
    address: <different>
    drivers: <same last name + same DOB + different address + different phone number>
    notes: <PATID_B almost has the same first name as PATID_A + same last name but different DOB and do not share the same address or phone number. Final verdict is I think they are different patients however my confidence is low.>

- Pair: PATID_A=<41A4634D5B113A0298C4DF66162D92B2> PATID_B=<99A27B3C403CD07CDC90DF44B19F2ED2> | section=§9.1 | score=<0.8631>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<816582D9813E6B3787A223DEEB480B65> PATID_B=<F087ECC809544465E28DE6F4DCDC3023> | section=§9.1 | score=<0.6156>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    notes: <no additional notes to report>

- Pair: PATID_A=<15FA264E76C3952E947781D288ED28E4> PATID_B=<E01448160E24677A9B8BFD9428765F89> | section=§9.1 | score=<0.7110>
    verdict: <unsure>
    confidence: <low>
    pattern: <other>
    address: <exact>
    drivers: <same first name + different last name + different DOB + exact address + same phone number>
    notes: <What is odd is these patient records have the same first name, same address and same phone number but are different in last name and DOB which would suggest they are different patients. Unable to verify on SSN because both patient records do not contain the SSN>

- Pair: PATID_A=<61D14D4FAB4072EBB6797AE46E9E8680> PATID_B=<E02F33DAC931CBF9865CEC004474BFA1> | section=§9.1 | score=<0.4733>
    verdict: <different>
    confidence: <high>
    pattern: <roommate-same-address>
    address: <exact>
    drivers: <different first name + different last name + same email + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<0FEEF5F76952F03CD2CC22AB60832A6C> PATID_B=<C9BA88D6EC666A621A1AE2FA85A67E64> | section=§9.1 | score=<0.4695>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>
   
- Pair: PATID_A=<B4D2F15EE6438D40BC025DAE43F934FD> PATID_B=<D93D21F1040898156C0E604B5AEED4AD> | section=§9.1 | score=<0.5360>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <same last name + same city + state + zip + different AL1>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<9C7079700F7E4642C9A6CB094C873B7F> PATID_B=<F7D8641948A22C485BD1A4DE17CB87C5> | section=§9.1 | score=<0.4958>
    verdict: <unsure>
    confidence: <medium>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <same first name + last name + almost same DOB + different AL1 + same city + state + zipcode + different phone number>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <The data is odd because the patient records have the same first name, last name, and the DOB only differs by one number in the month which could suggest a typo>

- Pair: PATID_A=<EF63FD2E24E598A9831899F9E7521EDA> PATID_B=<F5962686BD4EA1EE33368D543ED0F177> | section=§9.1 | score=<0.5487>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + different DOB + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<B3DA94C0BAD0F527AA64D2B680453EB2> PATID_B=<DB604069B623F8A94D8F32E47D580789> | section=§9.1 | score=<0.4570>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <same-household>
    drivers: <different first name + different last name + different DOB + same address + same phone number>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<73AD38C41B03EC774E70F44B94118767> PATID_B=<B0D4A5C82A7D4D2DC22E357BF1388D9B> | section=§9.1 | score=<0.4916>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>
   
- Pair: PATID_A=<4878430CF7F6D3E1136BAED7C60F539E> PATID_B=<FDD50488B5444FC49A755CD3F29C2236> | section=§9.1 | score=<0.4550>
    verdict: <unsure>
    confidence: <low>
    pattern: <same-person-name-change>
    address: <different>
    drivers: <same last name + same DOB + different address + different phone number>
    threshold (§9.2 only): <model-correct>
    notes: <the data is odd because the first name differs by one letter which suggests a typo given that the last names are the same and the DOB are the same>

- Pair: PATID_A=<1FBBB6AD96D0B386C5616BD4DA669179> PATID_B=<90266ACAA98E5DFDD1F6AE8B894D70E2> | section=§9.1 | score=<0.5437>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same email + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<6D7A610F463F4A27C8B496956905E288> PATID_B=<DB9FDAAC432C177B923691A172240204> | section=§9.1 | score=<0.5258>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <no additional notes to report>

- Pair: PATID_A=<A6A60A32B9C166AF79D62D20976CB9D1> PATID_B=<F7D8641948A22C485BD1A4DE17CB87C5> | section=§9.1 | score=<0.4958>
    verdict: <same>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <same-city-state-zip>
    drivers: <same first name + same last name + same city + state + zipcode>
    threshold (§9.2 only): <should-be-higher-tier>
    notes: <The DOB for the patient records differs by one digit in the month which could suggest a typo given that the first name and last name are the same>

- Pair: PATID_A=<3970FB9B7DB4610EFA41BB0742BC3F2A> PATID_B=<D6A64CCCF21D81E3A4F28D23629FBB2B> | section=§9.1 | score=<0.5157>
    verdict: <same>
    confidence: <low>
    pattern: <same person name change>
    address: <exact>
    drivers: <same first name + different last name + exact address + same phone + different DOB>
    threshold (§9.2 only): <model-correct | should-be-higher-tier | should-be-lower-tier>
    notes: <the difference in DOB could suggest they are different patients so my confidence is low>

- Pair: PATID_A=<4647BF4D0A5ABB5DEF5DBD3545C062D5> PATID_B=<C3258D1EC2E01D6F3147E6836A4E56A1> | section=§9.1 | score=<0.9153>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <no additional notes to report>

- Pair: PATID_A=<2A9E805B0E925849D85C06910D161140> PATID_B=<552AD9E5304E308AC243EEC20FA314D7> | section=§9.1 | score=<0.9309>
    verdict: <different>
    confidence: <high>
    pattern: <other>
    address: <different>
    drivers: <same first name + same last name + same DOB + different SSN>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <The model made a critical error here - if there is a difference in SSN this should be an automatic no match >

- Pair: PATID_A=<82B78923F1585C3D7B4B0DD6E85F5552> PATID_B=<9672BE0C5B5A917D43ED51CB6BA48207> | section=§9.1 | score=<0.8541>
    verdict: <different>
    confidence: <medium>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<A585399515E09F9482B9C8465B065783> PATID_B=<DC12DC8D542163619B9910EB59E28543> | section=§9.1 | score=<0.8937>
    verdict: <different>
    confidence: <high>
    pattern: <family-same-household>
    address: <exact>
    drivers: <different first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<00A75C5C27F1E9066A5C18D5FC7D557B> PATID_B=<D578816BF1D9EBD00FE64BF7434A5DC5> | section=§9.1 | score=<0.8721>
    verdict: <unsure>
    confidence: <medium>
    pattern: <roomates-same-address>
    address: <exact>
    drivers: <same first name + different last name + exact address + different DOB + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<BF69FBAEEAB134B1B619D0E6A18EAC6C> PATID_B=<D37991416B5D345FB781F2C7F5EEDFA1> | section=§9.1 | score=<0.8824>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <same first name + same last name + different DOB + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<1619504FF993089082820E778DD5935E> PATID_B=<52AE54209B94D7E37F6250617409DBA7> | section=§9.1 | score=<0.9409>
    verdict: <different>
    confidence: <high>>
    pattern: <other>
    address: <different>
    drivers: <different first name + same last name + different DOB + different AL1 + same city + state + zip + same phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<4F109F3C53C518D771F8E84AE03BE003> PATID_B=<CC9C232A6F3CEDE1B0B3868115D7FE8F> | section=§9.1 | score=<0.8526>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <different>
    drivers: <same first name + different last name +same DOB + different address + different phone number >
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<5E44CA62C42EFA5BE9579AEAEA5C5557> PATID_B=<CE548FF534B46CD262103AFFAEA6D8B7> | section=§9.1 | score=<0.9294>
    verdict: <different>
    confidence: <medium>
    pattern: <other>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + different DOB + different AL1 + same city + state + zip + different phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<54256AE52D2F6A9E7B3587407CE8437D> PATID_B=<6E764354E5CFAA4C860F6AD8378DAB88> | section=§9.1 | score=<0.9439>
    verdict: <same>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <exact>
    drivers: <different first name + same last name + exact address + same phone number>
    threshold (§9.2 only): <model-correct>
    notes: <>

- Pair: PATID_A=<00A346E3452250650488DCB33D563985> PATID_B=<06A61F36FBF53F381F214FACE8B56BA7> | section=§9.1 | score=<0.9311>
    verdict: <unsure>
    confidence: <high>
    pattern: <namesake-same-name>
    address: <same-city-state-zip>
    drivers: <different first name + same last name + same DOB + different AL1 + same city + state + zip + different phone number>
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

- Pair: PATID_A=<95D1836F915B9C831DF358C16A24D3F9> PATID_B=<ED3E769D0FF743DCBC66BCEEE31C3171> | section=§9.1 | score=<0.9392>
    verdict: <unsure>
    confidence: <medium>
    pattern: <same-person-name-change>
    address: <same-city-state-zip>
    drivers: <same first name + different last name + same DOB >
    threshold (§9.2 only): <should-be-lower-tier>
    notes: <>

